In [0]:
INSERT INTO retail_lakehouse.silver.sales
VALUES (9999, 1, 1001, 100, 2, CURRENT_DATE());

SELECT *
FROM table_changes('retail_lakehouse.silver.sales', 0);

UPDATE retail_lakehouse.silver.sales
SET Quantity = 5
WHERE TransactionID = 9999;

DELETE FROM retail_lakehouse.silver.sales
WHERE TransactionID = 9999;

In [0]:
ALTER TABLE retail_lakehouse.silver.sales
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

In [0]:
DESCRIBE HISTORY retail_lakehouse.silver.sales;

In [0]:
INSERT INTO retail_lakehouse.silver.sales
VALUES (8888, 1, 1001, 100, 3, CURRENT_DATE());

UPDATE retail_lakehouse.silver.sales
SET Quantity = 10
WHERE TransactionID = 8888;

DELETE FROM retail_lakehouse.silver.sales
WHERE TransactionID = 8888;

SELECT *
FROM table_changes('retail_lakehouse.silver.sales', 11);

In [0]:
MERGE INTO retail_lakehouse.gold.fact_sales t

USING (

    SELECT
        s.TransactionID,
        c.CustomerSK,
        p.ProductSK,
        st.StoreSK,
        s.Quantity,
        s.Quantity * p.UnitPrice AS Amount,
        s.TxnDate

    FROM table_changes('retail_lakehouse.silver.sales', 11) s

    JOIN retail_lakehouse.gold.dim_customer c
        ON s.CustomerID = c.CustomerID
        AND c.IsActive = TRUE

    JOIN retail_lakehouse.gold.dim_product p
        ON s.ProductID = p.ProductID

    JOIN retail_lakehouse.gold.dim_store st
        ON s.StoreID = st.StoreID

    WHERE s._change_type = 'insert'

) src

ON t.TransactionID = src.TransactionID

WHEN NOT MATCHED THEN
INSERT (
    SalesSK,
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)

VALUES (
    monotonically_increasing_id(),
    src.TransactionID,
    src.CustomerSK,
    src.ProductSK,
    src.StoreSK,
    src.Quantity,
    src.Amount,
    src.TxnDate
);

In [0]:
SELECT *
FROM retail_lakehouse.gold.fact_sales
ORDER BY TransactionID DESC;

**SCD type 2 working**

In [0]:
SELECT CustomerID, IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

SELECT
    CustomerID,
    StartDate,
    EndDate
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

In [0]:
SELECT COUNT(*)
FROM retail_lakehouse.gold.fact_sales;

INSERT INTO retail_lakehouse.silver.sales
VALUES (9999, 1, 1001, 100, 2, CURRENT_DATE());

SELECT TransactionID, COUNT(*)
FROM retail_lakehouse.gold.fact_sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;

In [0]:
UPDATE retail_lakehouse.silver.customers
SET City = 'Bangalore'
WHERE CustomerID = 1;
